# Environment and model setup

In [1]:
from pathlib import Path
import sys
import torch

# Project directory
PROJECT_ROOT = Path.cwd().parent

# Important directories
SRC_DIR = PROJECT_ROOT / "src"

ANTISPOOF_DIR = (
    PROJECT_ROOT / "third_party" / "face-anti-spoofing"
)

# Model checkpoints
RECOGNITION_CHECKPOINT = (
    PROJECT_ROOT / "checkpoints" / "epoch_checkpoint.pth"
)

ANTISPOOF_CHECKPOINT = (
    PROJECT_ROOT / "models" / "anti_spoofing" / "MiniFASNetV2.pth"
)


# Automatically select GPU or CPU
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce GTX 1650


# Import all three model architectures

In [2]:
# Project root
sys.path.insert(0, str(PROJECT_ROOT))

# MobileFaceNet + ArcFace
from src.face_recognition_model import FaceRecognitionModel

# MiniFASNetV2
sys.path.insert(
    0,
    str(PROJECT_ROOT / "third_party" / "face-anti-spoofing")
)
from models.fastnet import MiniFASNetV2

# RetinaFace
from uniface import RetinaFace

# Verify imports
print("MobileFaceNet + ArcFace:", FaceRecognitionModel)
print("\nMiniFASNetV2:", MiniFASNetV2)
print("\nRetinaFace:", RetinaFace)

MobileFaceNet + ArcFace: <class 'src.face_recognition_model.FaceRecognitionModel'>

MiniFASNetV2: <class 'models.fastnet.MiniFASNetV2'>

RetinaFace: <class 'uniface.detection.retinaface.RetinaFace'>


# Model initialization and checkpoint loading.

In [3]:
# Load MobileFaceNet + ArcFace
recognition_model = FaceRecognitionModel(
    embedding_size=128,
    num_classes=5103
).to(DEVICE)

recognition_checkpoint = torch.load(
    RECOGNITION_CHECKPOINT,
    map_location=DEVICE
)

recognition_model.load_state_dict(
    recognition_checkpoint["model_state_dict"]
)

recognition_model.eval()

# Load MiniFASNetV2
antispoof_model = MiniFASNetV2().to(DEVICE)

antispoof_checkpoint = torch.load(
    ANTISPOOF_CHECKPOINT,
    map_location=DEVICE
)

antispoof_model.load_state_dict(antispoof_checkpoint)

antispoof_model.eval()

# Initialize RetinaFace
face_detector = RetinaFace()


# checking is it actually in eval mode
print("Recognition model training mode:", recognition_model.training)
print("Anti-spoofing model training mode:", antispoof_model.training)


Recognition model training mode: False
Anti-spoofing model training mode: False


In [4]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms
from src.transforms import get_eval_transforms


# MobileFaceNet evaluation preprocessing
recognition_transform = get_eval_transforms()


def analyze_frame(image):

    results = []

    
    # Validate input image
    if image is None or image.size == 0:
        raise ValueError("Invalid input image.")

    # Detect faces
    detected_faces = face_detector.detect(image)

    if detected_faces is None or len(detected_faces) == 0:
        return results

    with torch.inference_mode():

        for face in detected_faces:

            bbox = np.asarray(face.bbox, dtype=np.float32)
            detection_confidence = float(face.confidence)

            # Bounding box coordinates
            x1, y1, x2, y2 = bbox.astype(int)

            # Expand bounding box around its centre
            width = x2 - x1
            height = y2 - y1

            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2

            crop_width = width * 2.7
            crop_height = height * 2.7

            left = max(0, int(center_x - crop_width / 2))
            top = max(0, int(center_y - crop_height / 2))

            right = min(
                image.shape[1],
                int(center_x + crop_width / 2)
            )

            bottom = min(
                image.shape[0],
                int(center_y + crop_height / 2)
            )

            face_crop = image[top:bottom, left:right]

            if face_crop.size == 0:
                continue

            # MiniFASNetV2 preprocessing: BGR, 80x80
            antispoof_image = cv2.resize(
                face_crop,
                (80, 80)
            )

            antispoof_image = (
                antispoof_image.astype(np.float32)
                .transpose(2, 0, 1)
            )

            antispoof_tensor = torch.from_numpy(
                antispoof_image
            ).unsqueeze(0).to(DEVICE)

            # Anti-spoofing inference
            antispoof_logits = antispoof_model(
                antispoof_tensor
            )

            antispoof_probabilities = torch.softmax(
                antispoof_logits,
                dim=1
            )[0]

            antispoof_class = int(
                torch.argmax(antispoof_probabilities).item()
            )

            antispoof_confidence = float(
                antispoof_probabilities[antispoof_class].item()
            )

            # Repository class mapping:
            # Class 1 = Real
            # Classes 0 and 2 = Fake
            is_real = antispoof_class == 1

            result = {
                "bbox": bbox.tolist(),
                "detection_confidence": detection_confidence,
                "is_real": is_real,
                "antispoof_class": antispoof_class,
                "antispoof_confidence": antispoof_confidence,
                "antispoof_probabilities": (
                    antispoof_probabilities.cpu().tolist()
                ),
                "embedding": None
            }

            # Only genuine faces proceed to recognition
            if is_real:

                # MobileFaceNet preprocessing: BGR -> RGB
                face_rgb = cv2.cvtColor(
                    face_crop,
                    cv2.COLOR_BGR2RGB
                )

                face_pil = Image.fromarray(face_rgb)

                recognition_tensor = recognition_transform(
                    face_pil
                ).unsqueeze(0).to(DEVICE)

                # Extract 128-dimensional embedding
                embedding = recognition_model.backbone(
                    recognition_tensor
                )

                # L2 normalization
                embedding = F.normalize(
                    embedding,
                    p=2,
                    dim=1
                )

                result["embedding"] = (
                    embedding.squeeze(0).cpu()
                )

            results.append(result)

    return results

In [5]:
image_path = Path(PROJECT_ROOT / "data" / "antispoofing" /"me.jpg")

image_bytes = np.fromfile(str(image_path), dtype=np.uint8)
image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

results = analyze_frame(image)

print("Detected faces:", len(results))

for i, result in enumerate(results):
    print(f"\nFace {i + 1}")
    print("Real face:", result["is_real"])
    print("Anti-spoof class:", result["antispoof_class"])
    print("Anti-spoof confidence:", result["antispoof_confidence"])
    print("Embedding shape:", result["embedding"].shape)

Detected faces: 1

Face 1
Real face: True
Anti-spoof class: 1
Anti-spoof confidence: 0.9479994177818298
Embedding shape: torch.Size([128])


# Face registration database

In [6]:
face_database = {}
print("Registered identities:", len(face_database))

Registered identities: 0


In [7]:
def register_identity(identity_name, images):

    if not identity_name or not identity_name.strip():
        raise ValueError("Identity name cannot be empty.")

    if identity_name in face_database:
        raise ValueError(f"{identity_name} is already registered.")

    if not images:
        raise ValueError("No images provided.")

    accepted_embeddings = []
    rejected_images = []

    for index, image in enumerate(images):

        results = analyze_frame(image)

        # Require exactly one detected face
        if len(results) != 1:
            rejected_images.append({
                "image": index + 1,
                "reason": "Expected exactly one face."
            })
            continue

        result = results[0]

        # Reject fake faces
        if not result["is_real"]:
            rejected_images.append({
                "image": index + 1,
                "reason": "Anti-spoofing classified it as fake."
            })
            continue

        # Store the genuine face embedding
        accepted_embeddings.append(
            result["embedding"].clone()
        )

    if not accepted_embeddings:
        print("Registration failed: No valid face embeddings.")
        return

    face_database[identity_name] = {
        "embeddings": accepted_embeddings,
        "num_embeddings": len(accepted_embeddings)
    }

    print(f"Registration completed for: {identity_name}")
    print("Accepted images:", len(accepted_embeddings))
    print("Rejected images:", len(rejected_images))

    for item in rejected_images:
        print(f"Image {item['image']}: {item['reason']}")

In [8]:
def capture_registration_images(num_images=6, delay=4):

    cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not cap.isOpened():
        raise RuntimeError("Unable to open webcam.")

    captured_images = []

    try:
        print("Face registration started!")
        print("Adjust your face whenever the countdown begins.\n")

        for i in range(num_images):

            # Countdown before capturing
            for remaining in range(delay, 0, -1):

                ret, frame = cap.read()

                if not ret:
                    continue

                preview = frame.copy()

                cv2.putText(
                    preview,
                    f"Capture {i + 1}/{num_images}",
                    (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    preview,
                    f"Capturing in: {remaining}",
                    (30, 100),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    (0, 0, 255),
                    3
                )

                cv2.imshow("Face Registration", preview)

                cv2.waitKey(1000)

            # Capture final frame
            ret, frame = cap.read()

            if ret:
                captured_images.append(frame.copy())
                print(f"Image {i + 1}/{num_images} captured!")

        print("\nRegistration capture completed!")

    finally:
        cap.release()
        cv2.destroyAllWindows()

    return captured_images

In [9]:
registration_images = capture_registration_images(6)

Face registration started!
Adjust your face whenever the countdown begins.

Image 1/6 captured!
Image 2/6 captured!
Image 3/6 captured!
Image 4/6 captured!
Image 5/6 captured!
Image 6/6 captured!

Registration capture completed!


In [10]:
print("Total captured:", len(registration_images))

Total captured: 6


In [11]:
# Register webcam captures under one identity
register_identity(
    identity_name="Yuvanesh_Webcam",
    images=registration_images
)

Registration completed for: Yuvanesh_Webcam
Accepted images: 6
Rejected images: 0


In [12]:
# Verify registered identity
identity = "Yuvanesh_Webcam"

if identity in face_database:

    embeddings = face_database[identity]["embeddings"]

    print("Identity:", identity)
    print("Total stored embeddings:", len(embeddings))
    print("Embedding shape:", embeddings[0].shape)

    print(
        "All embeddings:",
        torch.stack(embeddings).shape
    )

Identity: Yuvanesh_Webcam
Total stored embeddings: 6
Embedding shape: torch.Size([128])
All embeddings: torch.Size([6, 128])


# Create the authentication function

In [13]:
import torch.nn.functional as F

def authenticate_identity(embedding, face_database, threshold=0.9491418):

    if embedding is None:
        return {
            "authenticated": False,
            "identity": None,
            "similarity": None
        }

    best_identity = None
    best_similarity = -1.0

    query_embedding = F.normalize(
        embedding.reshape(1, -1),
        p=2,
        dim=1
    )

    for identity, data in face_database.items():

        registered_embeddings = torch.stack(
            data["embeddings"]
        ).to(embedding.device)

        registered_embeddings = F.normalize(
            registered_embeddings,
            p=2,
            dim=1
        )

        similarities = F.cosine_similarity(
            query_embedding,
            registered_embeddings,
            dim=1
        )

        identity_similarity = similarities.max().item()

        if identity_similarity > best_similarity:
            best_similarity = identity_similarity
            best_identity = identity

    authenticated = best_similarity >= threshold

    return {
        "authenticated": authenticated,
        "identity": best_identity if authenticated else None,
        "similarity": best_similarity
    }

In [14]:
# Capture a fresh webcam frame
cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

try:
    ret, test_frame = cap.read()
finally:
    cap.release()

if not ret:
    raise RuntimeError("Unable to capture webcam frame.")

# Analyze the captured frame
test_results = analyze_frame(test_frame)

if len(test_results) == 0:
    print("Authentication failed: No face detected.")

elif len(test_results) > 1:
    print("Authentication failed: Multiple faces detected.")

elif not test_results[0]["is_real"]:
    print("Authentication failed: Face classified as fake.")

else:
    # Authenticate the detected face
    auth_result = authenticate_identity(
        test_results[0]["embedding"],
        face_database
    )

    print("Authentication Result")
    print("---------------------")
    print("Authenticated:", auth_result["authenticated"])
    print("Identity:", auth_result["identity"])
    print("Cosine Similarity:", round(auth_result["similarity"], 4))

Authentication Result
---------------------
Authenticated: True
Identity: Yuvanesh_Webcam
Cosine Similarity: 0.9871


# src/face_engine.py

In [37]:
%%writefile "$PROJECT_ROOT/src/face_engine.py"

from pathlib import Path
import sys

import cv2
import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image

# --------------------------------------------------
# PROJECT PATHS
# --------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parent.parent

RECOGNITION_CHECKPOINT = (
    PROJECT_ROOT / "checkpoints" / "epoch_checkpoint.pth"
)

ANTISPOOF_CHECKPOINT = (
    PROJECT_ROOT / "models" / "anti_spoofing" / "MiniFASNetV2.pth"
)

ANTISPOOF_DIR = (
    PROJECT_ROOT / "third_party" / "face-anti-spoofing"
)

# Import third-party MiniFASNetV2 architecture
sys.path.insert(0, str(ANTISPOOF_DIR))

from models.fastnet import MiniFASNetV2

from uniface import RetinaFace
from src.face_recognition_model import FaceRecognitionModel
from src.transforms import get_eval_transforms


class FaceEngine:

    def __init__(self):

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # MobileFaceNet + ArcFace model
        self.recognition_model = FaceRecognitionModel(
            embedding_size=128,
            num_classes=5103
        ).to(self.device)

        recognition_checkpoint = torch.load(
            RECOGNITION_CHECKPOINT,
            map_location=self.device
        )

        self.recognition_model.load_state_dict(
            recognition_checkpoint["model_state_dict"]
        )

        self.recognition_model.eval()

        # MiniFASNetV2 anti-spoofing model
        self.antispoof_model = MiniFASNetV2().to(self.device)

        antispoof_checkpoint = torch.load(
            ANTISPOOF_CHECKPOINT,
            map_location=self.device
        )

        self.antispoof_model.load_state_dict(
            antispoof_checkpoint
        )

        self.antispoof_model.eval()

        # RetinaFace detector
        self.face_detector = RetinaFace()

        # MobileFaceNet evaluation preprocessing
        self.recognition_transform = get_eval_transforms()

        print("Face Engine initialized successfully.")
        print("Device:", self.device)

    # --------------------------------------------------
    # FACE ANALYSIS
    # RetinaFace + MiniFASNetV2 + MobileFaceNet
    # --------------------------------------------------

    def analyze_frame(self, image):

        results = []

        if image is None or image.size == 0:
            raise ValueError("Invalid input image.")

        detected_faces = self.face_detector.detect(image)

        if detected_faces is None or len(detected_faces) == 0:
            return results

        with torch.inference_mode():

            for face in detected_faces:

                bbox = np.asarray(
                    face.bbox,
                    dtype=np.float32
                )

                detection_confidence = float(face.confidence)

                x1, y1, x2, y2 = bbox.astype(int)

                # Expand bounding box by 2.7
                width = x2 - x1
                height = y2 - y1

                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2

                crop_width = width * 2.7
                crop_height = height * 2.7

                left = max(
                    0,
                    int(center_x - crop_width / 2)
                )

                top = max(
                    0,
                    int(center_y - crop_height / 2)
                )

                right = min(
                    image.shape[1],
                    int(center_x + crop_width / 2)
                )

                bottom = min(
                    image.shape[0],
                    int(center_y + crop_height / 2)
                )

                face_crop = image[top:bottom, left:right]

                if face_crop.size == 0:
                    continue

                # MiniFASNetV2 preprocessing
                # BGR, 80x80, float32, CHW
                antispoof_image = cv2.resize(
                    face_crop,
                    (80, 80)
                )

                antispoof_image = (
                    antispoof_image.astype(np.float32)
                    .transpose(2, 0, 1)
                )

                antispoof_tensor = torch.from_numpy(
                    antispoof_image
                ).unsqueeze(0).to(self.device)

                # Anti-spoofing inference
                antispoof_logits = self.antispoof_model(
                    antispoof_tensor
                )

                antispoof_probabilities = torch.softmax(
                    antispoof_logits,
                    dim=1
                )[0]

                antispoof_class = int(
                    torch.argmax(
                        antispoof_probabilities
                    ).item()
                )

                antispoof_confidence = float(
                    antispoof_probabilities[
                        antispoof_class
                    ].item()
                )

                # Repository class mapping:
                # Class 1 = Real
                # Classes 0 and 2 = Fake
                is_real = antispoof_class == 1

                result = {
                    "bbox": bbox.tolist(),
                    "detection_confidence": detection_confidence,
                    "is_real": is_real,
                    "antispoof_class": antispoof_class,
                    "antispoof_confidence": antispoof_confidence,
                    "antispoof_probabilities": (
                        antispoof_probabilities.cpu().tolist()
                    ),
                    "embedding": None
                }

                # Only genuine faces proceed to recognition
                if is_real:

                    # BGR -> RGB
                    face_rgb = cv2.cvtColor(
                        face_crop,
                        cv2.COLOR_BGR2RGB
                    )

                    face_pil = Image.fromarray(face_rgb)

                    recognition_tensor = (
                        self.recognition_transform(face_pil)
                        .unsqueeze(0)
                        .to(self.device)
                    )

                    # Extract 128D embedding
                    embedding = self.recognition_model.backbone(
                        recognition_tensor
                    )

                    embedding = F.normalize(
                        embedding,
                        p=2,
                        dim=1
                    )

                    result["embedding"] = (
                        embedding.squeeze(0).cpu()
                    )

                results.append(result)

        return results

    # --------------------------------------------------
    # FACE REGISTRATION
    # --------------------------------------------------

    def register_identity(
        self,
        identity_name,
        images,
        face_database
    ):

        if not identity_name or not identity_name.strip():
            raise ValueError("Identity name cannot be empty.")

        if identity_name in face_database:
            raise ValueError(
                f"{identity_name} is already registered."
            )

        if not images:
            raise ValueError("No registration images provided.")

        accepted_embeddings = []
        rejected_images = []

        for index, image in enumerate(images):

            results = self.analyze_frame(image)

            if len(results) != 1:
                rejected_images.append({
                    "image": index + 1,
                    "reason": "Expected exactly one face."
                })
                continue

            result = results[0]

            if not result["is_real"]:
                rejected_images.append({
                    "image": index + 1,
                    "reason": "Anti-spoofing classified it as fake."
                })
                continue

            accepted_embeddings.append(
                result["embedding"].clone()
            )

        if not accepted_embeddings:

            return {
                "success": False,
                "identity": identity_name,
                "accepted": 0,
                "rejected": rejected_images
            }

        face_database[identity_name] = {
            "embeddings": accepted_embeddings,
            "num_embeddings": len(accepted_embeddings)
        }

        return {
            "success": True,
            "identity": identity_name,
            "accepted": len(accepted_embeddings),
            "rejected": rejected_images
        }

    # --------------------------------------------------
    # IDENTITY AUTHENTICATION
    # --------------------------------------------------

    def authenticate_identity(
        self,
        embedding,
        face_database,
        threshold=0.9491418
    ):

        if embedding is None or not face_database:
            return {
                "authenticated": False,
                "identity": None,
                "similarity": None
            }

        query_embedding = F.normalize(
            embedding.reshape(1, -1),
            p=2,
            dim=1
        )

        best_identity = None
        best_similarity = -1.0

        for identity, data in face_database.items():

            registered_embeddings = torch.stack(
                data["embeddings"]
            ).to(embedding.device)

            registered_embeddings = F.normalize(
                registered_embeddings,
                p=2,
                dim=1
            )

            similarities = F.cosine_similarity(
                query_embedding,
                registered_embeddings,
                dim=1
            )

            # Average similarity across all registered templates
            identity_similarity = similarities.max().item()

            if identity_similarity > best_similarity:

                best_similarity = identity_similarity
                best_identity = identity

        authenticated = best_similarity >= threshold

        return {
            "authenticated": authenticated,
            "identity": best_identity if authenticated else None,
            "similarity": best_similarity
        }

    # --------------------------------------------------
    # COMBINED ACCESS DECISION
    # --------------------------------------------------

    def make_access_decision(
        self,
        analysis_results,
        face_database,
        threshold=0.9491418
    ):

        if len(analysis_results) == 0:
            return {
                "access_granted": False,
                "identity": None,
                "reason": "No face detected."
            }

        if len(analysis_results) > 1:
            return {
                "access_granted": False,
                "identity": None,
                "reason": "Multiple faces detected."
            }

        face = analysis_results[0]

        if not face["is_real"]:
            return {
                "access_granted": False,
                "identity": None,
                "reason": "Fake face detected."
            }

        auth_result = self.authenticate_identity(
            face["embedding"],
            face_database,
            threshold
        )

        if not auth_result["authenticated"]:
            return {
                "access_granted": False,
                "identity": None,
                "reason": "Identity not recognized.",
                "similarity": auth_result["similarity"]
            }

        return {
            "access_granted": True,
            "identity": auth_result["identity"],
            "reason": "Face verified successfully.",
            "similarity": auth_result["similarity"]
        }


Overwriting Y:\Python\AI ML\Month 2\week 8-NLP\47 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)/src/face_engine.py


# src/face_database.py

In [16]:
%%writefile "$PROJECT_ROOT/src/face_database.py"

import json
from pathlib import Path

import torch


# --------------------------------------------------
# DATABASE CONFIGURATION
# --------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parent.parent

DATABASE_DIR = PROJECT_ROOT / "data" / "registered_faces"


class FaceDatabase:

    def __init__(self):

        DATABASE_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        self.database_dir = DATABASE_DIR

    # --------------------------------------------------
    # VALIDATE IDENTITY NAME
    # --------------------------------------------------

    def _get_file_path(self, identity_name):

        identity_name = identity_name.strip()

        if not identity_name:
            raise ValueError("Identity name cannot be empty.")

        invalid_characters = '<>:"/\\|?*'

        if any(char in identity_name for char in invalid_characters):
            raise ValueError("Identity name contains invalid characters.")

        if identity_name in {".", ".."} or identity_name.endswith((" ", ".")):
            raise ValueError("Invalid identity name.")

        return self.database_dir / f"{identity_name}.json"

    # --------------------------------------------------
    # SAVE ONE IDENTITY
    # --------------------------------------------------

    def save_identity(self, identity_name, identity_data):

        file_path = self._get_file_path(identity_name)

        if file_path.exists():
            raise FileExistsError(
                f"{identity_name} is already stored."
            )

        embeddings = identity_data["embeddings"]

        if not embeddings:
            raise ValueError("No embeddings available to save.")

        embedding_values = [
            embedding.detach().cpu().reshape(-1).tolist()
            for embedding in embeddings
        ]

        if any(len(embedding) != 128 for embedding in embedding_values):
            raise ValueError("Every embedding must contain 128 values.")

        data = {
            "identity": identity_name,
            "embedding_size": 128,
            "num_embeddings": len(embedding_values),
            "embeddings": embedding_values
        }

        temporary_path = file_path.with_suffix(".tmp")

        try:
            with open(temporary_path, "w", encoding="utf-8") as file:
                json.dump(data, file, indent=4)

            temporary_path.replace(file_path)

        finally:
            if temporary_path.exists():
                temporary_path.unlink()

        print(f"Identity saved: {identity_name}")

    # --------------------------------------------------
    # LOAD ONE IDENTITY
    # --------------------------------------------------

    def load_identity(self, identity_name):

        file_path = self._get_file_path(identity_name)

        if not file_path.exists():
            raise FileNotFoundError(
                f"No registered identity found: {identity_name}"
            )

        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        embeddings = data["embeddings"]

        if data["embedding_size"] != 128:
            raise ValueError("Invalid embedding dimension.")

        if data["num_embeddings"] != len(embeddings):
            raise ValueError("Embedding count mismatch.")

        if not embeddings or any(len(item) != 128 for item in embeddings):
            raise ValueError("Invalid stored embeddings.")

        return {
            "embeddings": [
                torch.tensor(item, dtype=torch.float32)
                for item in embeddings
            ],
            "num_embeddings": len(embeddings)
        }

    # --------------------------------------------------
    # LOAD ALL REGISTERED IDENTITIES
    # --------------------------------------------------

    def load_database(self):

        face_database = {}

        for file_path in self.database_dir.glob("*.json"):

            with open(file_path, "r", encoding="utf-8") as file:
                data = json.load(file)

            identity_name = data["identity"]

            face_database[identity_name] = self.load_identity(
                identity_name
            )

        return face_database

    # --------------------------------------------------
    # LIST REGISTERED IDENTITIES
    # --------------------------------------------------

    def list_identities(self):

        return sorted(
            file_path.stem
            for file_path in self.database_dir.glob("*.json")
        )

    # --------------------------------------------------
    # DELETE ONE IDENTITY
    # --------------------------------------------------

    def delete_identity(self, identity_name):

        file_path = self._get_file_path(identity_name)

        if not file_path.exists():
            raise FileNotFoundError(
                f"No registered identity found: {identity_name}"
            )

        file_path.unlink()

        print(f"Identity deleted: {identity_name}")


Writing Y:\Python\AI ML\Month 2\week 8-NLP\47 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)/src/face_database.py


# app.py

In [25]:
%%writefile "$PROJECT_ROOT/app.py"

import cv2
import tkinter as tk

from tkinter import ttk, messagebox

from src.face_engine import FaceEngine
from src.face_database import FaceDatabase


# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

REGISTRATION_IMAGES = 6
CAPTURE_DELAY = 8
MATCH_THRESHOLD = 0.9491418


# --------------------------------------------------
# WEBCAM CAPTURE
# --------------------------------------------------

def capture_registration_images(num_images=6, delay=4):

    cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not cap.isOpened():
        raise RuntimeError("Unable to open webcam.")

    captured_images = []

    try:
        print("Registration started. Adjust your face for each capture.")

        for i in range(num_images):

            for remaining in range(delay, 0, -1):

                ret, frame = cap.read()

                if not ret:
                    raise RuntimeError("Unable to read webcam frame.")

                preview = frame.copy()

                cv2.putText(
                    preview,
                    f"Capture {i + 1}/{num_images}",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    preview,
                    f"Capturing in: {remaining}",
                    (30, 110),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    (0, 0, 255),
                    3
                )

                cv2.imshow("Face Registration", preview)

                if cv2.waitKey(1000) & 0xFF == 27:
                    return []

            ret, frame = cap.read()

            if not ret:
                raise RuntimeError("Unable to capture image.")

            captured_images.append(frame.copy())

            print(f"Captured image {i + 1}/{num_images}")

    finally:
        cap.release()
        cv2.destroyAllWindows()

    return captured_images


def capture_authentication_image():

    cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not cap.isOpened():
        raise RuntimeError("Unable to open webcam.")

    captured_image = None

    try:
        print("Press SPACE to capture your face. Press ESC to cancel.")

        while True:

            ret, frame = cap.read()

            if not ret:
                raise RuntimeError("Unable to read webcam frame.")

            preview = frame.copy()

            cv2.putText(
                preview,
                "Press SPACE to authenticate",
                (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.imshow("Face Authentication", preview)

            key = cv2.waitKey(1) & 0xFF

            if key == 32:
                captured_image = frame.copy()
                break

            elif key == 27:
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

    return captured_image


# --------------------------------------------------
# APPLICATION
# --------------------------------------------------

class FaceAuthenticationApp:

    def __init__(self, root):

        self.root = root
        self.root.title("Face Authentication System")
        self.root.geometry("650x750")
        self.root.resizable(True, True)

        # Initialize models
        self.engine = FaceEngine()

        # Initialize persistent database
        self.storage = FaceDatabase()
        self.face_database = self.storage.load_database()

        self.create_interface()
        self.refresh_identity_list()

    # --------------------------------------------------
    # INTERFACE
    # --------------------------------------------------

    def create_interface(self):

        main_frame = ttk.Frame(self.root, padding=25)
        main_frame.pack(fill="both", expand=True)

        ttk.Label(
            main_frame,
            text="FACE AUTHENTICATION SYSTEM",
            font=("Arial", 18, "bold")
        ).pack(pady=15)

        ttk.Label(
            main_frame,
            text="Face Registration and Identity Verification",
            font=("Arial", 10)
        ).pack(pady=5)

        # Registration section
        registration_frame = ttk.LabelFrame(
            main_frame,
            text="Register New Identity",
            padding=15
        )

        registration_frame.pack(fill="x", pady=15)

        self.name_entry = ttk.Entry(
            registration_frame,
            font=("Arial", 11)
        )

        self.name_entry.pack(
            fill="x",
            pady=8
        )

        self.name_entry.insert(0, "Enter identity name")

        ttk.Button(
            registration_frame,
            text="Register Face",
            command=self.register_face
        ).pack(pady=8)

        # Authentication section
        authentication_frame = ttk.LabelFrame(
            main_frame,
            text="Face Authentication",
            padding=15
        )

        authentication_frame.pack(fill="x", pady=10)

        ttk.Button(
            authentication_frame,
            text="Authenticate Using Webcam",
            command=self.authenticate_face
        ).pack(pady=8)

        # Result display
        self.status_label = ttk.Label(
            main_frame,
            text="System Ready",
            font=("Arial", 13, "bold")
        )

        self.status_label.pack(pady=15)

        # Registered identities
        identity_frame = ttk.LabelFrame(
            main_frame,
            text="Registered Identities",
            padding=10
        )

        identity_frame.pack(fill="both", expand=True, pady=10)

        self.identity_list = tk.Listbox(
            identity_frame,
            height=5,
            font=("Arial", 10)
        )

        self.identity_list.pack(
            fill="both",
            expand=True,
            pady=5
        )

        ttk.Button(
            identity_frame,
            text="Delete Selected Identity",
            command=self.delete_identity
        ).pack(pady=5)

    # --------------------------------------------------
    # REFRESH REGISTERED IDENTITIES
    # --------------------------------------------------

    def refresh_identity_list(self):

        self.identity_list.delete(0, tk.END)

        for identity in self.storage.list_identities():
            self.identity_list.insert(tk.END, identity)

    # --------------------------------------------------
    # REGISTER FACE
    # --------------------------------------------------

    def register_face(self):

        identity_name = self.name_entry.get().strip()

        if not identity_name or identity_name == "Enter identity name":
            messagebox.showwarning(
                "Registration",
                "Please enter an identity name."
            )
            return

        if identity_name in self.face_database:
            messagebox.showwarning(
                "Registration",
                "This identity is already registered."
            )
            return

        try:

            self.status_label.config(
                text="Capturing registration images...",
                foreground="blue"
            )

            self.root.update_idletasks()

            images = capture_registration_images(
                REGISTRATION_IMAGES,
                CAPTURE_DELAY
            )

            if not images:
                self.status_label.config(text="Registration cancelled.")
                return

            registration_result = self.engine.register_identity(
                identity_name,
                images,
                self.face_database
            )

            if not registration_result["success"]:

                self.status_label.config(
                    text="Registration failed.",
                    foreground="red"
                )

                messagebox.showerror(
                    "Registration Failed",
                    "No valid face embeddings were collected."
                )

                return

            # Persist the newly registered identity
            self.storage.save_identity(
                identity_name,
                self.face_database[identity_name]
            )

            self.refresh_identity_list()

            self.status_label.config(
                text=f"Registered: {identity_name}",
                foreground="green"
            )

            messagebox.showinfo(
                "Registration Successful",
                f"Identity: {identity_name}\n"
                f"Accepted images: {registration_result['accepted']}\n"
                f"Rejected images: {len(registration_result['rejected'])}"
            )

        except Exception as error:

            self.status_label.config(
                text="Registration error.",
                foreground="red"
            )

            messagebox.showerror(
                "Error",
                str(error)
            )

    # --------------------------------------------------
    # AUTHENTICATE FACE
    # --------------------------------------------------

    def authenticate_face(self):

        if not self.face_database:

            messagebox.showwarning(
                "Authentication",
                "No registered identities found."
            )
            return

        try:

            image = capture_authentication_image()

            if image is None:
                self.status_label.config(text="Authentication cancelled.")
                return

            analysis_results = self.engine.analyze_frame(image)

            access_result = self.engine.make_access_decision(
                analysis_results,
                self.face_database,
                threshold=MATCH_THRESHOLD
            )

            if access_result["access_granted"]:

                identity = access_result["identity"]
                similarity = access_result["similarity"]

                self.status_label.config(
                    text=f"ACCESS GRANTED: {identity}",
                    foreground="green"
                )

                messagebox.showinfo(
                    "Access Granted",
                    f"Identity: {identity}\n"
                    f"Cosine Similarity: {similarity:.4f}"
                )

            else:

                self.status_label.config(
                    text="ACCESS DENIED",
                    foreground="red"
                )

                messagebox.showwarning(
                    "Access Denied",
                    access_result["reason"]
                )

        except Exception as error:

            self.status_label.config(
                text="Authentication error.",
                foreground="red"
            )

            messagebox.showerror(
                "Error",
                str(error)
            )

    # --------------------------------------------------
    # DELETE IDENTITY
    # --------------------------------------------------

    def delete_identity(self):

        selection = self.identity_list.curselection()

        if not selection:
            messagebox.showwarning(
                "Delete Identity",
                "Please select an identity."
            )
            return

        identity_name = self.identity_list.get(selection[0])

        confirm = messagebox.askyesno(
            "Confirm Deletion",
            f"Delete {identity_name}?"
        )

        if not confirm:
            return

        try:

            self.storage.delete_identity(identity_name)

            self.face_database.pop(identity_name, None)

            self.refresh_identity_list()

            self.status_label.config(
                text=f"Deleted: {identity_name}",
                foreground="blue"
            )

        except Exception as error:

            messagebox.showerror(
                "Deletion Error",
                str(error)
            )


# --------------------------------------------------
# APPLICATION ENTRY POINT
# --------------------------------------------------

if __name__ == "__main__":

    root = tk.Tk()

    app = FaceAuthenticationApp(root)

    root.mainloop()


Overwriting Y:\Python\AI ML\Month 2\week 8-NLP\47 — APPLE FACE ID  (FACIAL RECOGNITION PROJECTS)/app.py


In [39]:
%run "$PROJECT_ROOT/app.py"

Face Engine initialized successfully.
Device: cuda
Press SPACE to capture your face. Press ESC to cancel.


In [27]:
from src.face_database import FaceDatabase

storage = FaceDatabase()

face_database = storage.load_database()

print("REGISTERED IDENTITIES")
print("---------------------")

for identity, data in face_database.items():
    print(
        identity,
        "| Embeddings:",
        data["num_embeddings"]
    )

print("\nTotal registered identities:", len(face_database))

REGISTERED IDENTITIES
---------------------
amma | Embeddings: 5

Total registered identities: 1
